In [30]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

In [31]:
load_dotenv(override=True)     # fuerza la sobrescritura

True

In [32]:
import os
from dotenv import dotenv_values

archivo = dotenv_values(".env").get("ANTHROPIC_API_KEY")
entorno = os.getenv("ANTHROPIC_API_KEY")

print("archivo:", repr(archivo[:14]), len(archivo))
print("entorno:", repr(entorno[:14]), len(entorno))
print("iguales:", archivo == entorno)

archivo: 'sk-ant-api03-V' 108
entorno: 'sk-ant-api03-V' 108
iguales: True


In [33]:
def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

In [34]:
#llm("Hey, what's up?")

Next session uv sync                  # reconstruye el entorno desde uv.lock
Hay que reactivar la api 



 # RAG Agent architecture

In [35]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [36]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [37]:
#print(prompt)

In [38]:
#question = "I just discovered the course. Can I join now?"
#answer = llm(prompt)
#print(answer)

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)    
    return llm(user_prompt)

In [40]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [41]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

In [44]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [ ]:
#minisearch is based on elasticsearch, but it is a lightweight version that can be used in memory without the need for a separate server. 
#It allows you to create an index of documents and perform searches on them.
#We can see the structure of the documents and the fields that we can use for indexing and searching in the cell above. 
# The documents have the following fields: "question", "section", "answer", and "course". We can use these fields to create an index and perform searches on them.
#Keyword fields are used for exact matches, while text fields are used for full-text search. In this case, we can use the "course" field as a keyword field and the "question", "section", and "answer" fields as text fields.
#keyword can help you to filter the docs by course, restricting the search to a specific course. This can be useful if you want to find answers to questions related to a specific course, rather than searching through all the courses.

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

In [ ]:
#search_results = index.search(question, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [49]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [50]:
search_results = search(question)